In [12]:
# ===========================
# 01. IMPORTS
# ===========================
import re
import time


# ===========================
# 02. PARAMETERS
# ===========================

# Default parameter values
DEFAULT_TABLE = "tb_op_fila_de_sinistros"

# Read input parameters with default fallback values.
table_name = oidlUtils.parameters.getParameter("table_name", DEFAULT_TABLE)

# Build the target table name.
target = f"lab_ailakehouse_gold.lab_digitalinsurance_aidp.{table_name}"

# Streaming checkpoint path.
gold_checkpoint_path = (
    f"/Volumes/lab_catalog_silver/digital_insurance/checkpoints/{table_name}_checkpoint_gold/"
)

In [13]:
# ===========================
# 03. HELPERS
# ===========================
def import_query(path):
    """Load the SQL query from a file."""
    with open(path, "r") as open_file:
        return open_file.read().strip().rstrip(";").rstrip()


def refresh_gold(_clock_df, batch_id):
    """Replace the current-state table in Autonomous AI Lakehouse."""
    started = time.monotonic()
    print(f"[batch {batch_id}] {target} refresh started.")

    # Load the business query for the current table.
    result = spark.sql(import_query(
        f"/Workspace/ai-data-platform/src/gold-ingestion/queries/"
        f"{table_name}.sql"
    ))

    # Write the refreshed snapshot through the external catalog.
    result.write.mode("overwrite").saveAsTable(target)

    print(
        f"[batch {batch_id}] {target} refresh completed - "
        f"{time.monotonic() - started:.1f}s."
    )

In [14]:
# ===========================
# 04. NEAR-REAL-TIME PROCESSING
# ===========================
# A clock stream is used because the queries read several Silver Delta
# tables and contain time windows that must expire even with no new rows.
df = (
    spark.readStream
         .format("rate")
         .option("rowsPerSecond", 1)
         .option("numPartitions", 1)
         .load()
)

# Process one micro-batch every ten minutes. The target writes performed
# by refresh_gold are batch writes through the external catalog connector.
stream = (
    df.writeStream
      .option("checkpointLocation",gold_checkpoint_path)
      .foreachBatch(refresh_gold)
      .trigger(processingTime="10 minutes")
      .queryName(f"gold_{table_name}_10m_refresh")
      .start()
)

stream.awaitTermination()

[batch 0] lab_ailakehouse_gold.lab_digitalinsurance_aidp.tb_op_fila_de_sinistros refresh started.


[batch 0] lab_ailakehouse_gold.lab_digitalinsurance_aidp.tb_op_fila_de_sinistros refresh completed - 36.6s.


[batch 1] lab_ailakehouse_gold.lab_digitalinsurance_aidp.tb_op_fila_de_sinistros refresh started.


[batch 1] lab_ailakehouse_gold.lab_digitalinsurance_aidp.tb_op_fila_de_sinistros refresh completed - 9.5s.


opc-request-id: csid6cb5015044c0b8a8148a2c0f070d/463b7766355f45e6b48f5a59ad38a0f5/1E50C22FE23B4DE5AA7B83CA98673CAE

Request is cancelled by the user.